In [ ]:

import os
import json
import time
import random
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import TransformerConv
from sklearn.metrics import roc_auc_score, average_precision_score

# =========================
# CONFIG (FAST vs STRICT)
# =========================
RUN_MODE = "strict"  # "fast" or "strict"

SEED = 42
LR = 1e-3
MIN_LR = 1e-5
WEIGHT_DECAY = 1e-5
HIDDEN_DIM = 96
HEADS = 2
NEG_POS_RATIO = 2.0
TOP_K = 50
RUN_ROOT = "runs_graph_transformer"
USE_AMP = True

# Loss / optimization
RANK_LOSS_WEIGHT = 0.2
LR_PLATEAU_PATIENCE = 4

if RUN_MODE == "strict":
    EPOCHS = 120
    # Larger coverage and no early stop for truly long training
    BASE_TRAIN_POS_SAMPLE = 300000
    BASE_MP_POS_SAMPLE = 400000
    MAX_TRAIN_POS_SAMPLE = 1200000
    MAX_MP_POS_SAMPLE = 1800000
    EVAL_POS_SAMPLE = None  # full split evaluation
    EVAL_BATCH_SIZE = 4096
    EARLY_STOPPING_PATIENCE = 10**9
    AP50_EVAL_PAIR_LIMIT = None  # evaluate on all pairs
else:
    EPOCHS = 60
    BASE_TRAIN_POS_SAMPLE = 150000
    BASE_MP_POS_SAMPLE = 200000
    MAX_TRAIN_POS_SAMPLE = 300000
    MAX_MP_POS_SAMPLE = 450000
    EVAL_POS_SAMPLE = 50000
    EVAL_BATCH_SIZE = 8192
    EARLY_STOPPING_PATIENCE = 12
    AP50_EVAL_PAIR_LIMIT = 10000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", DEVICE)
print("Run mode:", RUN_MODE)

run_name = datetime.now().strftime("run_%Y%m%d_%H%M%S")
run_dir = os.path.join(RUN_ROOT, f"{RUN_MODE}_{run_name}")
os.makedirs(run_dir, exist_ok=True)
print("Run directory:", run_dir)

wall_start = time.time()

# =========================
# LOAD GRAPH + LABEL MATRIX
# =========================
graph_data = torch.load("graph_data.pt", map_location="cpu")
Y_bin = torch.load("Y_pair.pt", map_location="cpu").long()

x = graph_data.x.float().to(DEVICE)
num_pairs = int(graph_data.num_pairs)
num_labels = int(graph_data.num_labels)

if Y_bin.shape != (num_pairs, num_labels):
    raise ValueError(f"Y shape {tuple(Y_bin.shape)} != ({num_pairs}, {num_labels})")

# positive edges (pair_idx, label_idx)
pos_edges = torch.nonzero(Y_bin == 1, as_tuple=False)
num_pos = pos_edges.shape[0]
print("Positive pair-label edges:", num_pos)

perm = torch.randperm(num_pos)
pos_edges = pos_edges[perm]

n_train = int(0.8 * num_pos)
n_val = int(0.1 * num_pos)
train_pos = pos_edges[:n_train]
val_pos = pos_edges[n_train:n_train + n_val]
test_pos = pos_edges[n_train + n_val:]

print("Train positives:", train_pos.shape[0])
print("Val positives:", val_pos.shape[0])
print("Test positives:", test_pos.shape[0])

Y_bool = Y_bin.bool()


def sample_positive_subset(pos_tensor, n_max):
    if n_max is None or pos_tensor.shape[0] <= n_max:
        return pos_tensor
    idx = torch.randperm(pos_tensor.shape[0])[:n_max]
    return pos_tensor[idx]


def build_mp_edge_index(pos_edge_tensor):
    pair_nodes = pos_edge_tensor[:, 0]
    label_nodes = pos_edge_tensor[:, 1] + num_pairs

    src = torch.cat([pair_nodes, label_nodes], dim=0)
    dst = torch.cat([label_nodes, pair_nodes], dim=0)
    edge_index = torch.stack([src, dst], dim=0)
    return edge_index.to(DEVICE)


def sample_random_negatives(pos_edge_tensor, ratio=1.0):
    n_neg = int(pos_edge_tensor.shape[0] * ratio)
    neg_pairs = torch.randint(0, num_pairs, (n_neg,))
    neg_labels = torch.randint(0, num_labels, (n_neg,))

    collision = Y_bool[neg_pairs, neg_labels]
    while collision.any():
        m = int(collision.sum().item())
        neg_pairs[collision] = torch.randint(0, num_pairs, (m,))
        neg_labels[collision] = torch.randint(0, num_labels, (m,))
        collision = Y_bool[neg_pairs, neg_labels]

    return torch.stack([neg_pairs, neg_labels], dim=1)


class GraphTransformer(nn.Module):
    def __init__(self, dim, hidden_dim=96, heads=2, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(dim, hidden_dim)
        self.conv1 = TransformerConv(hidden_dim, hidden_dim, heads=heads, concat=False, dropout=dropout)
        self.conv2 = TransformerConv(hidden_dim, hidden_dim, heads=1, concat=False, dropout=dropout)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index):
        x = self.in_proj(x)
        x = self.conv1(x, edge_index)
        x = self.norm1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        x = self.norm2(x)
        return x


class LinkPredictor(nn.Module):
    def __init__(self, node_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(node_dim * 2, node_dim),
            nn.ReLU(),
            nn.Linear(node_dim, 1),
        )

    def forward(self, pair_emb, label_emb):
        z = torch.cat([pair_emb, label_emb], dim=-1)
        return self.mlp(z).squeeze(-1)


@torch.no_grad()
def get_full_node_embeddings(encoder, mp_edge_index):
    encoder.eval()
    with torch.autocast(device_type="cuda", enabled=(USE_AMP and DEVICE.type == "cuda")):
        h = encoder(x, mp_edge_index)
    return h


@torch.no_grad()
def sample_hard_negatives(encoder, predictor, pos_edge_tensor, mp_edge_index, n_hard):
    if n_hard <= 0:
        return torch.empty((0, 2), dtype=torch.long)

    # Candidate pool from random negatives, then select highest model-score negatives.
    candidate = sample_random_negatives(pos_edge_tensor, ratio=3.0)
    h = get_full_node_embeddings(encoder, mp_edge_index)

    logits_chunks = []
    for s in range(0, candidate.shape[0], EVAL_BATCH_SIZE):
        e = min(s + EVAL_BATCH_SIZE, candidate.shape[0])
        batch = candidate[s:e].to(DEVICE)
        pair_h = h[batch[:, 0]]
        label_h = h[num_pairs + batch[:, 1]]
        logits = predictor(pair_h, label_h)
        logits_chunks.append(logits.detach().cpu())

    logits_all = torch.cat(logits_chunks, dim=0)
    top_idx = torch.topk(logits_all, k=min(n_hard, logits_all.shape[0]), dim=0).indices
    return candidate[top_idx]


@torch.no_grad()
def evaluate_split(encoder, predictor, pos_split, mp_edge_index, neg_ratio=1.0):
    pos_eval = sample_positive_subset(pos_split, EVAL_POS_SAMPLE)
    neg_eval = sample_random_negatives(pos_eval, ratio=neg_ratio)

    h = get_full_node_embeddings(encoder, mp_edge_index)

    def score_edges(edge_tensor):
        scores = []
        for s in range(0, edge_tensor.shape[0], EVAL_BATCH_SIZE):
            e = min(s + EVAL_BATCH_SIZE, edge_tensor.shape[0])
            batch = edge_tensor[s:e].to(DEVICE)
            pair_h = h[batch[:, 0]]
            label_h = h[num_pairs + batch[:, 1]]
            logits = predictor(pair_h, label_h)
            scores.append(torch.sigmoid(logits).detach().cpu())
        return torch.cat(scores, dim=0)

    pos_prob = score_edges(pos_eval)
    neg_prob = score_edges(neg_eval)

    y_true = torch.cat([
        torch.ones_like(pos_prob),
        torch.zeros_like(neg_prob),
    ]).numpy()
    y_prob = torch.cat([pos_prob, neg_prob]).numpy()

    auc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    return auc, auprc


@torch.no_grad()
def compute_ap_at_k(encoder, predictor, mp_edge_index, y_true_matrix, k=50, eval_pair_limit=10000):
    encoder.eval()
    predictor.eval()

    h = get_full_node_embeddings(encoder, mp_edge_index)
    pair_h = h[:num_pairs]
    label_h = h[num_pairs:]

    pos_per_pair = y_true_matrix.sum(dim=1)
    candidate_pairs = torch.where(pos_per_pair > 0)[0]
    if eval_pair_limit is not None:
        candidate_pairs = candidate_pairs[:eval_pair_limit]

    ap_values = []
    for p in candidate_pairs.tolist():
        p_vec = pair_h[p].unsqueeze(0).repeat(num_labels, 1)
        logits = predictor(p_vec, label_h)
        probs = torch.sigmoid(logits)

        topk = torch.topk(probs, k=min(k, num_labels), dim=0).indices.cpu()
        rel = y_true_matrix[p, topk].float()

        if rel.sum() == 0:
            continue

        cum_rel = torch.cumsum(rel, dim=0)
        ranks = torch.arange(1, rel.shape[0] + 1, dtype=torch.float32)
        precision_at_i = cum_rel / ranks
        ap = (precision_at_i * rel).sum() / min(float(y_true_matrix[p].sum().item()), float(k))
        ap_values.append(ap.item())

    return float(np.mean(ap_values)) if ap_values else float("nan")


def curriculum_value(epoch, total_epochs, base_val, max_val):
    # linear warmup of sample sizes
    progress = (epoch - 1) / max(1, total_epochs - 1)
    return int(base_val + progress * (max_val - base_val))


# =========================
# TRAINING
# =========================
encoder = GraphTransformer(dim=x.shape[1], hidden_dim=HIDDEN_DIM, heads=HEADS, dropout=0.1).to(DEVICE)
predictor = LinkPredictor(node_dim=HIDDEN_DIM).to(DEVICE)

optimizer = torch.optim.AdamW(
    list(encoder.parameters()) + list(predictor.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=LR_PLATEAU_PATIENCE,
    min_lr=MIN_LR,
)

best_val_auc = -1.0
best_state = None
bad_epochs = 0
epoch_logs = []

scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    encoder.train()
    predictor.train()

    train_sample_n = curriculum_value(epoch, EPOCHS, BASE_TRAIN_POS_SAMPLE, MAX_TRAIN_POS_SAMPLE)
    mp_sample_n = curriculum_value(epoch, EPOCHS, BASE_MP_POS_SAMPLE, MAX_MP_POS_SAMPLE)

    train_pos_epoch = sample_positive_subset(train_pos, train_sample_n)
    mp_pos_epoch = sample_positive_subset(train_pos, mp_sample_n)
    mp_edge_index = build_mp_edge_index(mp_pos_epoch)

    # mixed negatives: random + hard
    n_neg = int(train_pos_epoch.shape[0] * NEG_POS_RATIO)
    n_hard = n_neg // 2
    n_rand = n_neg - n_hard

    rand_neg = sample_random_negatives(train_pos_epoch, ratio=(n_rand / max(1, train_pos_epoch.shape[0])))
    hard_neg = sample_hard_negatives(encoder, predictor, train_pos_epoch, mp_edge_index, n_hard)
    train_neg = torch.cat([rand_neg, hard_neg], dim=0).to(DEVICE)

    train_pos_dev = train_pos_epoch.to(DEVICE)
    edge_pairs = torch.cat([train_pos_dev, train_neg], dim=0)
    labels = torch.cat([
        torch.ones(train_pos_dev.shape[0], device=DEVICE),
        torch.zeros(train_neg.shape[0], device=DEVICE),
    ], dim=0)

    with torch.autocast(device_type="cuda", enabled=(USE_AMP and DEVICE.type == "cuda")):
        h = encoder(x, mp_edge_index)
        pair_h = h[edge_pairs[:, 0]]
        label_h = h[num_pairs + edge_pairs[:, 1]]
        logits = predictor(pair_h, label_h)

        bce_loss = F.binary_cross_entropy_with_logits(logits, labels)

        # pairwise ranking term on matched pos/neg subset
        n_rank = min(train_pos_dev.shape[0], train_neg.shape[0])
        pos_logits_rank = predictor(h[train_pos_dev[:n_rank, 0]], h[num_pairs + train_pos_dev[:n_rank, 1]])
        neg_logits_rank = predictor(h[train_neg[:n_rank, 0]], h[num_pairs + train_neg[:n_rank, 1]])
        rank_loss = F.margin_ranking_loss(
            pos_logits_rank,
            neg_logits_rank,
            target=torch.ones_like(pos_logits_rank),
            margin=0.2,
        )

        loss = bce_loss + RANK_LOSS_WEIGHT * rank_loss

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    val_auc, val_auprc = evaluate_split(encoder, predictor, val_pos, mp_edge_index, neg_ratio=1.0)
    scheduler.step(val_auc)

    current_lr = float(optimizer.param_groups[0]["lr"])
    row = {
        "epoch": epoch,
        "train_loss": float(loss.item()),
        "bce_loss": float(bce_loss.item()),
        "rank_loss": float(rank_loss.item()),
        "val_auc": float(val_auc),
        "val_auprc": float(val_auprc),
        "lr": current_lr,
        "train_sample_n": int(train_sample_n),
        "mp_sample_n": int(mp_sample_n),
        "epoch_seconds": float(time.time() - epoch_start),
    }
    epoch_logs.append(row)

    improved = val_auc > best_val_auc
    if improved:
        best_val_auc = val_auc
        bad_epochs = 0
        best_state = {
            "encoder": {k: v.detach().cpu().clone() for k, v in encoder.state_dict().items()},
            "predictor": {k: v.detach().cpu().clone() for k, v in predictor.state_dict().items()},
            "best_epoch": epoch,
            "best_val_auc": float(val_auc),
            "best_val_auprc": float(val_auprc),
        }
        torch.save(best_state, os.path.join(run_dir, "best_model.pt"))
    else:
        bad_epochs += 1

    if epoch == 1 or epoch % 2 == 0:
        print(
            f"Epoch {epoch:02d} | loss={loss.item():.4f} | "
            f"val_auc={val_auc:.4f} | val_auprc={val_auprc:.4f} | lr={current_lr:.2e} | "
            f"sample(train/mp)={train_sample_n}/{mp_sample_n} | "
            f"epoch_time={row['epoch_seconds']:.1f}s"
        )

    if bad_epochs >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping at epoch {epoch} (no improvement for {EARLY_STOPPING_PATIENCE} checks).")
        break

if best_state is None:
    raise RuntimeError("Training failed: no best checkpoint was captured.")

encoder.load_state_dict(best_state["encoder"])
predictor.load_state_dict(best_state["predictor"])
encoder.to(DEVICE)
predictor.to(DEVICE)

# =========================
# FINAL EVALUATION
# =========================
final_mp_edge_index = build_mp_edge_index(sample_positive_subset(train_pos, MAX_MP_POS_SAMPLE))

test_auc, test_auprc = evaluate_split(
    encoder,
    predictor,
    test_pos,
    final_mp_edge_index,
    neg_ratio=1.0,
)

ap50 = compute_ap_at_k(
    encoder,
    predictor,
    final_mp_edge_index,
    y_true_matrix=Y_bin,
    k=TOP_K,
    eval_pair_limit=AP50_EVAL_PAIR_LIMIT,
)

print("\n===== FINAL METRICS (V2) =====")
print(f"AUC   : {test_auc:.4f}")
print(f"AUPRC : {test_auprc:.4f}")
print(f"AP@50 : {ap50:.4f}")

# save final model and artifacts
final_ckpt = {
    "encoder": encoder.state_dict(),
    "predictor": predictor.state_dict(),
    "config": {
        "seed": SEED,
        "hidden_dim": HIDDEN_DIM,
        "heads": HEADS,
        "epochs": EPOCHS,
        "lr": LR,
        "min_lr": MIN_LR,
        "weight_decay": WEIGHT_DECAY,
        "neg_pos_ratio": NEG_POS_RATIO,
        "top_k": TOP_K,
        "rank_loss_weight": RANK_LOSS_WEIGHT,
    },
    "split_sizes": {
        "train_pos": int(train_pos.shape[0]),
        "val_pos": int(val_pos.shape[0]),
        "test_pos": int(test_pos.shape[0]),
    },
    "best_epoch": int(best_state["best_epoch"]),
    "best_val_auc": float(best_state["best_val_auc"]),
    "best_val_auprc": float(best_state["best_val_auprc"]),
    "test_metrics": {
        "auc": float(test_auc),
        "auprc": float(test_auprc),
        "ap_at_50": float(ap50),
    },
}

torch.save(final_ckpt, "graph_transformer_linkpred_v2.pt")
torch.save(final_ckpt, os.path.join(run_dir, "final_model.pt"))

log_csv_path = os.path.join(run_dir, "train_log.csv")
with open(log_csv_path, "w", encoding="utf-8") as f:
    f.write("epoch,train_loss,bce_loss,rank_loss,val_auc,val_auprc,lr,train_sample_n,mp_sample_n\n")
    for row in epoch_logs:
        f.write(
            f"{row['epoch']},{row['train_loss']:.8f},{row['bce_loss']:.8f},{row['rank_loss']:.8f},"
            f"{row['val_auc']:.8f},{row['val_auprc']:.8f},{row['lr']:.10f},{row['train_sample_n']},{row['mp_sample_n']}\n"
        )

with open(os.path.join(run_dir, "train_log.json"), "w", encoding="utf-8") as f:
    json.dump(epoch_logs, f, indent=2)

metrics_payload = {
    "run_mode": RUN_MODE,
    "best_epoch": int(best_state["best_epoch"]),
    "best_val_auc": float(best_state["best_val_auc"]),
    "best_val_auprc": float(best_state["best_val_auprc"]),
    "test_auc": float(test_auc),
    "test_auprc": float(test_auprc),
    "test_ap_at_50": float(ap50),
    "total_wall_minutes": float((time.time() - wall_start) / 60.0),
}
with open(os.path.join(run_dir, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

total_minutes = (time.time() - wall_start) / 60.0
print("Saved checkpoint: graph_transformer_linkpred_v2.pt")
print("Saved best checkpoint:", os.path.join(run_dir, "best_model.pt"))
print("Saved final checkpoint:", os.path.join(run_dir, "final_model.pt"))
print("Saved logs:", log_csv_path)
print("Saved metrics:", os.path.join(run_dir, "metrics.json"))
print(f"Total wall time: {total_minutes:.2f} minutes")

c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  import torch_geometric.typing
c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  import torch_geometric.typing
c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-spline-conv'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  import torch_geometric.typing
c:\Users\mg276\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An iss

Training device: cuda
Run mode: strict
Run directory: runs_graph_transformer_v2\strict_run_20260420_161454


C:\Users\mg276\AppData\Local\Temp\ipykernel_24832\125582404.py:77: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  graph_data = torch.load("graph_data.pt", map_location="cpu")

Positive pair-label edges: 4560334
Train positives: 3648267
Val positives: 456033
Test positives: 456034
Epoch 01 | loss=0.7241 | val_auc=0.5420 | val_auprc=0.5118 | lr=1.00e-03 | sample(train/mp)=300000/400000 | epoch_time=1.1s
Epoch 02 | loss=0.6889 | val_auc=0.6030 | val_auprc=0.5651 | lr=1.00e-03 | sample(train/mp)=307563/411764 | epoch_time=0.8s
Epoch 04 | loss=0.6729 | val_auc=0.6401 | val_auprc=0.6155 | lr=1.00e-03 | sample(train/mp)=322689/435294 | epoch_time=1.1s
Epoch 06 | loss=0.6720 | val_auc=0.6401 | val_auprc=0.6207 | lr=1.00e-03 | sample(train/mp)=337815/458823 | epoch_time=1.0s
Epoch 08 | loss=0.6720 | val_auc=0.6601 | val_auprc=0.6407 | lr=1.00e-03 | sample(train/mp)=352941/482352 | epoch_time=1.1s
Epoch 10 | loss=0.6706 | val_auc=0.6788 | val_auprc=0.6594 | lr=1.00e-03 | sample(train/mp)=368067/505882 | epoch_time=1.1s
Epoch 12 | loss=0.6704 | val_auc=0.6826 | val_auprc=0.6640 | lr=1.00e-03 | sample(train/mp)=383193/529411 | epoch_time=1.1s
Epoch 14 | loss=0.6688 | va